# Input perturbation analysis

Tests whether different encoder architectures correctly attribute control inputs (Base, Mod boluses)
to mechanistically relevant parameters. 

Each model is run open-loop on manually-defined bolus sequences. The resulting θ_k trajectories
reveal whether the model conflates a known control input with a parameter change — and whether
different architectures disagree on *which* parameters respond.

**Mechanistic expectations (mof_synthesis_4):**

| Bolus | Should move | Should stay flat |
|---|---|---|
| Base only | k_base, k_nuc_A, k_gro_A, k_nuc_C, k_gro_C | k_mod, K_I |
| Mod only | k_mod, K_I | k_base, k_nuc_A, k_gro_A, k_nuc_C, k_gro_C |

Ideally θ stays **flat** — the model knows the perturbation came from u_k, not from a parameter change.
Deviations from flat, especially cross-channel, are identifiability failures.

In [1]:
import sys
from pathlib import Path

# Find last-layer-ode/ regardless of where the notebook is launched from
_here = Path().resolve()
_llo = next(
    (p for p in [_here, _here.parent, _here.parent.parent]
     if (p / 'plot_diagnostics.py').exists()),
    _here,
)
if str(_llo) not in sys.path:
    sys.path.insert(0, str(_llo))

PROJECT_ROOT = _llo.parent  # theta-lab/
print(f'project root: {PROJECT_ROOT}')

import csv
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from plot_diagnostics import rebuild_model_from_experiment, device_auto

project root: /gpfs/home2/overven1/theta-lab


KeyboardInterrupt: 

## Config — edit this cell to change which runs are compared

In [ ]:
# ── One entry per architecture ─────────────────────────────────────────────
# Paths are relative to the project root (theta-lab/).
RUN_DIRS: dict[str, str] = {
    "GRU": "experiments/mof_synthesis_baselines/2026-03-26/20260326_163723_mof_synthesis_4",
    # "Transformer": "experiments/.../...",
    # "Mamba":       "experiments/maba_test/...",
}

OUT_DIR = PROJECT_ROOT / "results/input_perturbation/mof4_arch_comparison"

# Bolus amplitudes — median nonzero values from training data
BASE_AMP = 1.0
MOD_AMP  = 0.55

# Pulse positions (step indices, 0-based, out of K=299 total steps)
BASE_STEPS = [30, 80, 130, 180, 230]
MOD_STEPS  = [30, 80, 130, 180, 230]
# ──────────────────────────────────────────────────────────────────────────

## Load models

In [ ]:
device = device_auto()
print(f'Device: {device}')

models        = {}
ref_ds        = None
theta_names   = None
control_names = None

for label, run_path in RUN_DIRS.items():
    exp_dir = PROJECT_ROOT / run_path
    model, ds, state_names, th_names = rebuild_model_from_experiment(exp_dir, device)
    model.eval()
    models[label] = model
    print(f'  {label}: theta_dim={model.theta_dim}, P={model.P}')
    if ref_ds is None:
        ref_ds        = ds
        theta_names   = th_names
        control_names = ds.control_names.tolist() if hasattr(ds, 'control_names') else ['Base', 'Mod']

K     = ref_ds.u_seq.shape[1]
U     = ref_ds.u_seq.shape[2]
dt_np = ref_ds.dt                              # (K,) — already np.diff(t_obs)
t_mid = np.cumsum(dt_np) - dt_np / 2          # midpoint time of each step

print(f'\nK={K}, U={U}')
print(f'theta params ({len(theta_names)}): {theta_names}')
print(f'control inputs: {control_names}')

Device: mps
  GRU: theta_dim=7, P=4

K=299, U=2
theta params (7): ['θ0', 'θ1', 'θ2', 'θ3', 'θ4', 'θ5', 'θ6']
control inputs: ['Base', 'Mod']


## Define bolus scenarios

In [ ]:
def make_u_seq(base_steps, mod_steps, base_amp, mod_amp, K, U):
    """Build a (1, K, U) control tensor with boluses at specified step indices."""
    u = torch.zeros(1, K, U)
    for s in base_steps:
        if 0 <= s < K:
            u[0, s, 0] = base_amp
    for s in mod_steps:
        if 0 <= s < K:
            u[0, s, 1] = mod_amp
    return u

SCENARIOS = {
    'base_only':   make_u_seq(BASE_STEPS, [],        BASE_AMP, MOD_AMP,  K, U),
    'mod_only':    make_u_seq([],         MOD_STEPS,  BASE_AMP, MOD_AMP, K, U),
    'interleaved': make_u_seq(BASE_STEPS[::2], MOD_STEPS[1::2], BASE_AMP, MOD_AMP, K, U),
    'both':        make_u_seq(BASE_STEPS, MOD_STEPS,  BASE_AMP, MOD_AMP, K, U),
}

# Visualise the designed sequences
fig, axes = plt.subplots(len(SCENARIOS), 1, figsize=(12, 2.5 * len(SCENARIOS)), sharex=True)
colors = ['steelblue', 'tomato']
for ax, (name, u_seq) in zip(axes, SCENARIOS.items()):
    for ci, cname in enumerate(control_names):
        vals = u_seq[0, :, ci].numpy()
        ax.step(t_mid, vals, where='mid', color=colors[ci], label=cname, linewidth=1.5)
    ax.set_title(name, fontsize=10)
    ax.set_ylabel('bolus')
    ax.legend(fontsize=8)
fig.suptitle('Designed U sequences', fontsize=12)
fig.tight_layout()
plt.show()

/var/folders/kn/bdxct8j50cjfj1n9y9t057200000gn/T/ipykernel_80306/1566200738.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Run open-loop rollouts

In [ ]:
def rollout(model, u_seq, dt_np, device):
    """Open-loop rollout from y0=0. Returns (y_out, theta_out) as numpy arrays."""
    B, K, U = u_seq.shape
    P = model.P
    y0     = torch.zeros(B, P, device=device)
    u_t    = u_seq.to(device)
    dt_t   = torch.tensor(dt_np, device=device).unsqueeze(0).expand(B, -1)
    obs_idx = torch.arange(P, device=device)
    with torch.no_grad():
        y_out, theta_out, _ = model(
            y0, u_t, dt_t, obs_idx,
            y_seq=None, teacher_forcing=False,
        )
    return y_out[0].cpu().numpy(), theta_out[0].cpu().numpy()  # (K, P), (K, theta_dim)

# results[scenario][label] = {'y': (K,P), 'theta': (K, theta_dim)}
results = {}
for scenario, u_seq in SCENARIOS.items():
    results[scenario] = {}
    for label, model in models.items():
        y_out, theta_out = rollout(model, u_seq, dt_np, device)
        results[scenario][label] = {'y': y_out, 'theta': theta_out}
        print(f'  {scenario} | {label}  theta range: [{theta_out.min():.3f}, {theta_out.max():.3f}]')

  base_only | GRU  theta range: [0.001, 2.000]
  mod_only | GRU  theta range: [0.001, 2.000]
  interleaved | GRU  theta range: [0.001, 2.000]
  both | GRU  theta range: [0.001, 2.000]


## Export CSVs

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

for scenario, model_results in results.items():
    csv_path = OUT_DIR / f'{scenario}_theta.csv'
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['step', 'time', 'model'] + theta_names)
        for label, data in model_results.items():
            for k in range(K):
                writer.writerow(
                    [k, round(float(t_mid[k]), 4), label]
                    + [round(float(v), 6) for v in data['theta'][k]]
                )
    print(f'Saved {csv_path}')

# Also save state trajectories
for scenario, model_results in results.items():
    csv_path = OUT_DIR / f'{scenario}_states.csv'
    with open(csv_path, 'w', newline='') as f:
        obs_names = ref_ds.obs_names.tolist() if hasattr(ref_ds, 'obs_names') else [f'y{i}' for i in range(model.P)]
        writer = csv.writer(f)
        writer.writerow(['step', 'time', 'model'] + obs_names)
        for label, data in model_results.items():
            for k in range(K):
                writer.writerow(
                    [k, round(float(t_mid[k]), 4), label]
                    + [round(float(v), 6) for v in data['y'][k]]
                )
    print(f'Saved {csv_path}')

Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/base_only_theta.csv
Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/mod_only_theta.csv
Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/interleaved_theta.csv
Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/both_theta.csv
Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/base_only_states.csv
Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/mod_only_states.csv
Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/interleaved_states.csv
Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/both_states.csv


## Plot: θ trajectories per scenario
Each subplot is one theta parameter. Lines = architectures. Vertical dashed lines = bolus events.
Ideally lines are flat — deviations mean the model is attributing the bolus to a parameter change.

In [ ]:
MODEL_COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']
BOLUS_COLORS = {'Base': 'steelblue', 'Mod': 'tomato'}

def get_bolus_times(u_seq, t_mid, control_names):
    """Return {control_name: [times]} for all nonzero bolus events."""
    events = {}
    for ci, cname in enumerate(control_names):
        steps = np.where(u_seq[0, :, ci].numpy() > 0)[0]
        events[cname] = t_mid[steps].tolist()
    return events

n_theta = len(theta_names)
ncols   = 3
nrows   = (n_theta + ncols - 1) // ncols

for scenario, model_results in results.items():
    u_seq   = SCENARIOS[scenario]
    boluses = get_bolus_times(u_seq, t_mid, control_names)

    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows), sharex=True)
    axes_flat = axes.flatten()

    for ti, pname in enumerate(theta_names):
        ax = axes_flat[ti]
        for li, (label, data) in enumerate(model_results.items()):
            ax.plot(t_mid, data['theta'][:, ti],
                    color=MODEL_COLORS[li], label=label, linewidth=1.2)
        # Mark bolus events
        for cname, times in boluses.items():
            for t in times:
                ax.axvline(t, color=BOLUS_COLORS.get(cname, 'gray'),
                           alpha=0.35, linewidth=1.0, linestyle='--')
        ax.set_title(pname, fontsize=10)
        ax.set_ylabel('θ', fontsize=9)
        ax.grid(True, alpha=0.2)

    # Hide unused subplots
    for ti in range(n_theta, len(axes_flat)):
        axes_flat[ti].set_visible(False)

    # Legend: models
    model_handles = [mpatches.Patch(color=MODEL_COLORS[i], label=l)
                     for i, l in enumerate(model_results)]
    # Legend: bolus colours
    bolus_handles = [mpatches.Patch(color=BOLUS_COLORS.get(cn, 'gray'), alpha=0.5, label=f'{cn} bolus')
                     for cn in control_names if boluses.get(cn)]
    fig.legend(handles=model_handles + bolus_handles,
               loc='lower center', ncol=len(model_results) + len(bolus_handles),
               fontsize=9, bbox_to_anchor=(0.5, -0.01))

    axes_flat[-ncols].set_xlabel('time')
    fig.suptitle(f'θ trajectories — {scenario}', fontsize=13, y=1.01)
    fig.tight_layout()

    plot_path = OUT_DIR / f'{scenario}_theta.png'
    fig.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {plot_path}')

/var/folders/kn/bdxct8j50cjfj1n9y9t057200000gn/T/ipykernel_80306/243199701.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/base_only_theta.png
Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/mod_only_theta.png
Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/interleaved_theta.png
Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/both_theta.png


## Plot: state trajectories
Sanity check — do the rollouts produce physically plausible states? Different models should agree
broadly on the state trajectories even if their θ estimates diverge.

In [ ]:
obs_names = ref_ds.obs_names.tolist() if hasattr(ref_ds, 'obs_names') else [f'y{i}' for i in range(model.P)]

for scenario, model_results in results.items():
    u_seq   = SCENARIOS[scenario]
    boluses = get_bolus_times(u_seq, t_mid, control_names)
    P       = len(obs_names)
    ncols_s = min(P, 4)
    nrows_s = (P + ncols_s - 1) // ncols_s

    fig, axes = plt.subplots(nrows_s, ncols_s, figsize=(13, 3 * nrows_s), sharex=True)
    axes_flat = np.array(axes).flatten()

    for si, sname in enumerate(obs_names):
        ax = axes_flat[si]
        for li, (label, data) in enumerate(model_results.items()):
            ax.plot(t_mid, data['y'][:, si],
                    color=MODEL_COLORS[li], label=label, linewidth=1.2)
        for cname, times in boluses.items():
            for t in times:
                ax.axvline(t, color=BOLUS_COLORS.get(cname, 'gray'),
                           alpha=0.35, linewidth=1.0, linestyle='--')
        ax.set_title(sname, fontsize=10)
        ax.set_ylabel('concentration', fontsize=8)
        ax.grid(True, alpha=0.2)

    for si in range(P, len(axes_flat)):
        axes_flat[si].set_visible(False)

    fig.legend(handles=[mpatches.Patch(color=MODEL_COLORS[i], label=l)
                        for i, l in enumerate(model_results)],
               loc='lower center', ncol=len(model_results), fontsize=9,
               bbox_to_anchor=(0.5, -0.01))
    axes_flat[-ncols_s].set_xlabel('time')
    fig.suptitle(f'State trajectories — {scenario}', fontsize=13, y=1.01)
    fig.tight_layout()

    plot_path = OUT_DIR / f'{scenario}_states.png'
    fig.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {plot_path}')

/var/folders/kn/bdxct8j50cjfj1n9y9t057200000gn/T/ipykernel_80306/1659193858.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/base_only_states.png
Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/mod_only_states.png
Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/interleaved_states.png
Saved /Users/olivervanerven/Documents/Thesis/theta-lab/results/input_perturbation/mof4_arch_comparison/both_states.png


## θ response at bolus events
For each scenario, shows the mean absolute change in each θ in the 3 steps following a bolus,
relative to the pre-bolus baseline. A large value means the model reacted to that bolus
by adjusting that parameter.

In [ ]:
def theta_response_at_bolus(theta_traj, bolus_steps, window=3):
    """Mean absolute change in θ in [step, step+window) relative to [step-window, step)."""
    responses = []
    for s in bolus_steps:
        pre  = s - window
        post = s + window
        if pre < 0 or post >= theta_traj.shape[0]:
            continue
        baseline = theta_traj[max(0, pre):s, :].mean(axis=0)
        after    = theta_traj[s:post, :].mean(axis=0)
        responses.append(np.abs(after - baseline) / (np.abs(baseline) + 1e-8))
    return np.array(responses).mean(axis=0) if responses else np.zeros(theta_traj.shape[1])

for scenario, model_results in results.items():
    u_seq = SCENARIOS[scenario]
    print(f'\n── {scenario} ──')
    header = f'  {"":20s}' + ''.join(f'  {n[:10]:>10s}' for n in theta_names)
    print(header)
    for cname in control_names:
        ci = control_names.index(cname)
        bolus_steps = np.where(u_seq[0, :, ci].numpy() > 0)[0].tolist()
        if not bolus_steps:
            continue
        print(f'  {cname} bolus:')
        for label, data in model_results.items():
            resp = theta_response_at_bolus(data['theta'], bolus_steps)
            vals = ''.join(f'  {v:>10.3f}' for v in resp)
            print(f'    {label:18s}{vals}')


── base_only ──
                                θ0          θ1          θ2          θ3          θ4          θ5          θ6
  Base bolus:
    GRU                      0.000       0.058       0.022       0.047       0.078       0.043       0.068

── mod_only ──
                                θ0          θ1          θ2          θ3          θ4          θ5          θ6
  Mod bolus:
    GRU                      0.000       0.057       0.014       0.046       0.078       0.042       0.067

── interleaved ──
                                θ0          θ1          θ2          θ3          θ4          θ5          θ6
  Base bolus:
    GRU                      0.000       0.097       0.016       0.077       0.076       0.072       0.060
  Mod bolus:
    GRU                      0.000       0.000       0.016       0.000       0.080       0.000       0.079

── both ──
                                θ0          θ1          θ2          θ3          θ4          θ5          θ6
  Base bolus:
    GRU     

## Scenario overlay — is the model sensitive to *which* input was applied?
Plots `base_only` and `mod_only` theta trajectories on the same axes per parameter.
If the model correctly attributes causality, the lines should diverge. If they overlap, the model is blind to which input arrived.

In [ ]:
SCENARIO_STYLES = {
    'base_only':   {'color': 'steelblue', 'ls': '-',  'label': 'Base only'},
    'mod_only':    {'color': 'tomato',    'ls': '-',  'label': 'Mod only'},
    'both':        {'color': 'seagreen',  'ls': '--', 'label': 'Both'},
    'interleaved': {'color': 'orchid',    'ls': ':',  'label': 'Interleaved'},
}

for label in models:
    n_theta = len(theta_names)
    ncols   = 3
    nrows   = (n_theta + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows), sharex=True)
    axes_flat = axes.flatten()

    for ti, pname in enumerate(theta_names):
        ax = axes_flat[ti]
        for scenario, style in SCENARIO_STYLES.items():
            if scenario not in results:
                continue
            theta_traj = results[scenario][label]['theta']
            ax.plot(t_mid, theta_traj[:, ti],
                    color=style['color'], ls=style['ls'],
                    label=style['label'], linewidth=1.5, alpha=0.85)
        ax.set_title(pname, fontsize=10)
        ax.set_ylabel('θ', fontsize=9)
        ax.grid(True, alpha=0.2)

    for ti in range(n_theta, len(axes_flat)):
        axes_flat[ti].set_visible(False)

    handles = [plt.Line2D([0], [0], color=s['color'], ls=s['ls'], label=s['label'])
               for sc, s in SCENARIO_STYLES.items() if sc in results]
    fig.legend(handles=handles, loc='lower center', ncol=len(handles),
               fontsize=10, bbox_to_anchor=(0.5, -0.01))
    axes_flat[-ncols].set_xlabel('time')
    fig.suptitle(f'θ per scenario — {label}  (overlaid)', fontsize=13, y=1.01)
    fig.tight_layout()

    plot_path = OUT_DIR / f'scenario_overlay_{label}.png'
    fig.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {plot_path}')